# Full-tissue napari visualization

Whole-tissue overview with all cell centers, then progressive polygon boundary loading.

In [1]:
from pathlib import Path

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Qt/OpenGL settings for Ubuntu + Wayland
os.environ["QT_QPA_PLATFORM"] = "xcb"
os.environ["PYOPENGL_PLATFORM"] = "glx"

import napari

In [2]:
PROJECT = Path.home() / "Projects/xenium_lung"
VIS_DIR = PROJECT / "results/xenium_5k/visualization"

COORDINATES_FILE = VIS_DIR / "cell_coordinates.parquet"
CLUSTERS_FILE = VIS_DIR / "cluster_labels.csv"
BOUNDARIES_FILE = VIS_DIR / "cell_boundaries.parquet"

In [3]:
coordinates = pd.read_parquet(COORDINATES_FILE)
clusters = pd.read_csv(CLUSTERS_FILE)

cells = coordinates.merge(
    clusters,
    on="cell_id",
    how="left",
    validate="one_to_one"
)

print("Cells:", cells.shape)
print(cells.head())

Cells: (266179, 4)
      cell_id           x            y  group
0  aaaaadnb-1  822.469055  5111.537598      0
1  aaaabalp-1  843.901428  5149.261230      0
2  aaaadfei-1  831.219421  5133.179199      1
3  aaaadjia-1  839.742981  5159.693848      0
4  aaaafglb-1  784.250916  5143.785645      1


In [4]:
print("Number of cells:", len(cells))
print("Unique cell IDs:", cells["cell_id"].is_unique)

print("Missing X:", cells["x"].isna().sum())
print("Missing Y:", cells["y"].isna().sum())
print("Missing cluster:", cells["group"].isna().sum())

print("\nX range:")
print(cells["x"].min(), "→", cells["x"].max())

print("\nY range:")
print(cells["y"].min(), "→", cells["y"].max())

print("\nClusters:")
print(sorted(cells["group"].dropna().unique()))

Number of cells: 266179
Unique cell IDs: True
Missing X: 0
Missing Y: 0
Missing cluster: 0

X range:
9.384281158447266 → 11467.05078125

Y range:
145.14759826660156 → 7783.51318359375

Clusters:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19)]


# Full-tissue point visualization

In [5]:
# napari uses (Y, X) rather than (X, Y)
all_points = cells[["y", "x"]].to_numpy(dtype=np.float32)

print("Point array shape:", all_points.shape)
print("Point array dtype:", all_points.dtype)

Point array shape: (266179, 2)
Point array dtype: float32


In [6]:
cluster_ids = sorted(cells["group"].dropna().unique())

cmap = plt.get_cmap("tab20", len(cluster_ids))

cluster_colors = {
    cluster_id: cmap(i)
    for i, cluster_id in enumerate(cluster_ids)
}

point_colors = np.array([
    cluster_colors[group]
    if not pd.isna(group)
    else (0.5, 0.5, 0.5, 1.0)
    for group in cells["group"]
])

print("Number of colors:", len(point_colors))
print("Color array shape:", point_colors.shape)

Number of colors: 266179
Color array shape: (266179, 4)


In [7]:
viewer = napari.Viewer()

In [8]:
viewer.add_points(
    all_points,
    size=2,
    face_color=point_colors,
    name="All cells — clusters",
)

<Points layer 'All cells — clusters' at 0x7a3eb84d3490>

In [9]:
viewer.layers["All cells — clusters"].size = 1.5

# Add full cell boundaries progressively

In [10]:
boundaries = pd.read_parquet(BOUNDARIES_FILE)

print("Boundary table:", boundaries.shape)
print(boundaries.head())
print("Index name:", boundaries.index.name)

Boundary table: (266179, 1)
                                                      geometry
cell_id_str                                                   
aaaaadnb-1   b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x19\x00...
aaaabalp-1   b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x19\x00...
aaaadfei-1   b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x19\x00...
aaaadjia-1   b"\x01\x03\x00\x00\x00\x01\x00\x00\x00\x19\x00...
aaaafglb-1   b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x19\x00...
Index name: cell_id_str


In [11]:
cluster_lookup = cells.set_index("cell_id")["group"]

boundary_groups = np.array([
    cluster_lookup.get(str(cell_id), np.nan)
    for cell_id in boundaries.index
])

print("Boundary groups:", len(boundary_groups))
print("Missing boundary groups:", pd.isna(boundary_groups).sum())

Boundary groups: 266179
Missing boundary groups: 0


# Progressive polygon loading

Do not run 266k polygons yet. Start with 5,000.

In [12]:
N_POLYGONS = 5_000

boundaries_test = boundaries.iloc[:N_POLYGONS]

print("Testing with:", len(boundaries_test), "polygons")

Testing with: 5000 polygons


In [13]:
from shapely import from_wkb

geometries = from_wkb(
    boundaries_test["geometry"].to_numpy()
)

print("Geometries:", len(geometries))
print("First geometry type:", geometries[0].geom_type)

Geometries: 5000
First geometry type: Polygon


In [14]:
polygon_data = []
polygon_groups = []

for cell_id, geom in zip(boundaries_test.index, geometries):

    group = cluster_lookup.get(str(cell_id), np.nan)

    if geom.geom_type == "Polygon":

        coords = np.asarray(geom.exterior.coords)

        # X,Y → Y,X
        coords = coords[:, [1, 0]]

        polygon_data.append(coords)
        polygon_groups.append(group)

    elif geom.geom_type == "MultiPolygon":

        for polygon in geom.geoms:

            coords = np.asarray(polygon.exterior.coords)

            # X,Y → Y,X
            coords = coords[:, [1, 0]]

            polygon_data.append(coords)
            polygon_groups.append(group)

print("Number of polygons:", len(polygon_data))

Number of polygons: 5000


In [15]:
polygon_groups = np.array(polygon_groups)

edge_colors = np.array([
    cluster_colors[group]
    if not pd.isna(group)
    else (0.5, 0.5, 0.5, 1.0)
    for group in polygon_groups
])

print("Edge colors:", edge_colors.shape)

Edge colors: (5000, 4)


In [16]:
viewer.add_shapes(
    polygon_data,
    shape_type="polygon",
    edge_width=0.8,
    edge_color=edge_colors,
    face_color=[0, 0, 0, 0],
    name=f"Cell boundaries — {N_POLYGONS:,}",
)

<Shapes layer 'Cell boundaries — 5,000' at 0x7a3e58c065d0>

## Reusable function to add a boundary subset

If 5,000 works smoothly, try `n_polygons=10_000`, then `25_000`, then `50_000`. Do not jump directly to 266,179.

In [ ]:
def add_boundary_subset(
    viewer,
    boundaries,
    cluster_lookup,
    cluster_colors,
    n_polygons=5000,
    layer_name=None,
):
    from shapely import from_wkb

    subset = boundaries.iloc[:n_polygons]

    geometries = from_wkb(
        subset["geometry"].to_numpy()
    )

    polygon_data = []
    polygon_groups = []

    for cell_id, geom in zip(subset.index, geometries):

        group = cluster_lookup.get(str(cell_id), np.nan)

        if geom.geom_type == "Polygon":

            coords = np.asarray(geom.exterior.coords)
            coords = coords[:, [1, 0]]

            polygon_data.append(coords)
            polygon_groups.append(group)

        elif geom.geom_type == "MultiPolygon":

            for polygon in geom.geoms:

                coords = np.asarray(polygon.exterior.coords)
                coords = coords[:, [1, 0]]

                polygon_data.append(coords)
                polygon_groups.append(group)

    edge_colors = np.array([
        cluster_colors[group]
        if not pd.isna(group)
        else (0.5, 0.5, 0.5, 1.0)
        for group in polygon_groups
    ])

    if layer_name is None:
        layer_name = f"Cell boundaries — {n_polygons:,}"

    return viewer.add_shapes(
        polygon_data,
        shape_type="polygon",
        edge_width=0.8,
        edge_color=edge_colors,
        face_color=[0, 0, 0, 0],
        name=layer_name,
    )

In [20]:
layer = add_boundary_subset(
    viewer,
    boundaries,
    cluster_lookup,
    cluster_colors,
    n_polygons=100_000,
)

## Note on representativeness

The first rows of the parquet are not necessarily spatially representative. For a proper full-tissue boundary visualization, consider a spatially distributed subset or a zoom-dependent approach (few/no polygons zoomed out, full resolution zoomed in), rather than forcing napari to render 266k polygons at once.

Next notebook: `05_napari_morphology.ipynb`, bringing in the Xenium morphology/H&E image aligned with these coordinates and boundaries.